In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
# Define source_file and table_name
source_file = f"{landing_folder_path}/{v_batch_id}/results"
table_name = f"{catalog_name}.{bronze_schema}.results"

**Step 1 - Read the CSV file using the dataframe reader API**

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, StringType, DateType, FloatType, IntegerType

results_schema = StructType([
    StructField("date", DateType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", IntegerType()),
    StructField("url", StringType()),
    StructField("constructorId", StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", FloatType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType()),
])

In [0]:
results_df = (
    spark.read
         .format("json")
         .schema(results_schema)
         .option('mode','FAILFAST')
         .load(source_file)
)

In [0]:
display(results_df)

**Step 2 - Add Metadata Columns**

- Source File
- Ingestion Timestamp

In [0]:
results_final_df = add_ingestion_metadata(results_df)

In [0]:
display(results_final_df)

**Step 3 - Write to bronze delta table**

In [0]:
write_to_bronze(
    input_df = results_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
%sql
select season, count(*)
from formula1_incr.bronze.results
group by season
order by season